This pipeline test uses:
- **Count and area:** YOLOv8-seg (Segmentation)
- **Stage Classificator:** Custom CNN
- **Action Motor:** Strict Rules

In [1]:
from pathlib import Path
import sys

In [2]:
project_root = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "utils" / "utls_install_libs.py").is_file()),
    None,
)

if project_root is None:
    raise FileNotFoundError("Could not find utils/utls_install_libs.py")

sys.path.insert(0, str(project_root))

In [3]:
from utils.utls_install_libs import install_missing_libs
from utils.utls_work_images import build_dataset
from utils.utls_ia_models import build_fast_cnn

In [4]:
install_missing_libs(['ultralytics'])

Checking project dependencies...
✔️ ultralytics pronta.


In [5]:
import time
import cv2
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.8 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\heito\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [6]:
print("📥 [1/4] Baixando matrizes do Google Drive...")
X_exp, y_exp = build_dataset(limit_per_class_collected=50, folder_name_collected="expulsion")
X_inc, y_inc = build_dataset(limit_per_class_collected=50, folder_name_collected="incubation")

X_completo = np.concatenate((X_exp, X_inc), axis=0)
y_completo = np.concatenate((y_exp, y_inc), axis=0)
X_train, X_test, y_train, y_test = train_test_split(X_completo, y_completo, test_size=0.2, random_state=42)

📥 [1/4] Baixando matrizes do Google Drive...


In [7]:
print("🧠 [2/4] Preparando a CNN Customizada e baixando YOLOv8n-seg...")
# Classificador Leve (CNN do Pipeline 01)
modelo_reator = build_fast_cnn()
modelo_reator.fit(X_train, y_train, epochs=5, verbose=0)

# Segmentador Pesado (YOLOv8 Nano Segmentation)
# O Keras fará o download do arquivo .pt automaticamente
modelo_yolo = YOLO('yolov8n-seg.pt')

🧠 [2/4] Preparando a CNN Customizada e baixando YOLOv8n-seg...


C:\Users\heito\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [8]:
def calcular_area_bolhas_yolo(img_array, modelo_yolo):
    img_uint8 = (img_array * 255).astype(np.uint8)
    
    # Inferência do YOLO (verbose=False oculta os logs por imagem)
    resultados = modelo_yolo.predict(img_uint8, verbose=False)
    
    qtd_bolhas = 0
    area_vapor = 0
    
    if resultados[0].masks is not None:
        # Conta quantas máscaras (objetos segmentados) o modelo encontrou
        qtd_bolhas = len(resultados[0].masks.data)
        
        # Soma a área em pixels de cada máscara
        mascaras = resultados[0].masks.data.cpu().numpy()
        for mascara in mascaras:
            # Padroniza o redimensionamento da máscara para 224x224
            mascara_resized = cv2.resize(mascara, (224, 224))
            area_vapor += np.sum(mascara_resized > 0.5)
            
    # Trava o limite em 100% para evitar flutuações de sobreposição de máscaras
    percentual_vapor = min((area_vapor / (224 * 224)) * 100, 100.0)
    return qtd_bolhas, round(percentual_vapor, 2)

In [9]:
def motor_de_regras(classe_predita, percentual_vapor):
    # Classe 0 = Expulsão, Classe 1 = Incubação
    if classe_predita == 0:
        if percentual_vapor > 30:
            return "CRÍTICO: Expulsão severa. Acionar resfriamento de emergência máximo."
        return "ALERTA ALTO: Expulsão detectada. Aumentar fluxo da bomba."
    else:
        if percentual_vapor > 15:
            return "ALERTA MÉDIO: Incubação acelerada. Monitorar temperatura."
        return "NORMAL: Incubação inicial. Manter operação."

In [10]:
resultados_p4 = []
print("⏱️ [3/4] Iniciando cronometragem do Pipeline 04 (YOLOv8-seg + CNN)...")

for i in range(len(X_test)):
    img = X_test[i]
    inicio_timer = time.perf_counter()
    
    # Estágio 1: Contagem e Área por Deep Learning (YOLOv8-seg)
    qtd_bolhas, pct_vapor = calcular_area_bolhas_yolo(img, modelo_yolo)
    
    # Estágio 2: Classificação de Estado (Sua CNN)
    img_batch = np.expand_dims(img, axis=0) 
    probabilidade = modelo_reator.predict(img_batch, verbose=0)[0][0]
    classe_ia = 1 if probabilidade > 0.5 else 0 
    
    # Estágio 3: Decisão Regras
    acao_sugerida = motor_de_regras(classe_ia, pct_vapor)
    
    fim_timer = time.perf_counter()
    latencia_ms = round((fim_timer - inicio_timer) * 1000, 2)
    
    resultados_p4.append({
        "ID_Teste": i,
        "Real": "Incubação" if y_test[i] == 1 else "Expulsão",
        "Predicao_CNN": "Incubação" if classe_ia == 1 else "Expulsão",
        "Qtd_Bolhas": qtd_bolhas,
        "Vapor_%": pct_vapor,
        "Acao_Sugerida": acao_sugerida,
        "Latencia_ms": latencia_ms
    })

⏱️ [3/4] Iniciando cronometragem do Pipeline 04 (YOLOv8-seg + CNN)...


In [11]:
df_p4 = pd.DataFrame(resultados_p4)
df_p4.to_csv('benchmark_pipeline_04.csv', index=False)

print(f"\n✅ [4/4] Benchmark concluído! Salvo como 'benchmark_pipeline_04.csv'.")
print(f"📊 Latência Média do Pipeline 04: {df_p4['Latencia_ms'].mean():.2f} ms")
display(df_p4.head())


✅ [4/4] Benchmark concluído! Salvo como 'benchmark_pipeline_04.csv'.
📊 Latência Média do Pipeline 04: 633.24 ms


,ID_Teste,Real,Predicao_CNN,Qtd_Bolhas,Vapor_%,Acao_Sugerida,Latencia_ms
0,0,Incubação,Incubação,0,0.0,NORMAL: Incubação inicial. Manter operação.,6847.19
1,1,Incubação,Incubação,0,0.0,NORMAL: Incubação inicial. Manter operação.,366.88
2,2,Incubação,Incubação,0,0.0,NORMAL: Incubação inicial. Manter operação.,327.88
3,3,Expulsão,Expulsão,0,0.0,ALERTA ALTO: Expulsão detectada. Aumentar flux...,339.23
4,4,Expulsão,Expulsão,0,0.0,ALERTA ALTO: Expulsão detectada. Aumentar flux...,310.99
